# Equal-Weight S&P 500 Index Fund

## Introduction & Library Imports

The S&P 500 is the world's most popular stock market index. The largest fund that is benchmarked to this index is the SPDR® S&P 500® ETF Trust. It has more than US$250 billion of assets under management.

The goal of this section of the course is to create a Python script that will accept the value of your portfolio and tell you how many shares of each S&P 500 constituent you should purchase to get an equal-weight version of the index fund.

## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [1]:
import numpy as np
import pandas as pd
import requests
import math
import xlsxwriter


# Prepare the Notebook
## Set the Environment

Select which workflow the notebook should run by setting `switch` to a value from `1` to `5`:

- **1 — demo**: Run EODHD API calls with the `demo` API key for a limited symbol set (`AAPL.US`, `AMZN.US`, `TSLA.US`).
- **2 — train**: Train and save the machine learning model only *(if model training is included in this notebook)*.
- **3 — test**: Load and test the machine learning model only *(if model testing is included in this notebook)*.
- **4 — external**: Use an offline external data source; no API calls are required (all data comes from CSV files).
- **5 — production**: Run the full production workflow, including complete API calls and, if applicable, loading and running the machine learning model.

In [2]:
# Select the workflow mode:
# 1 = demo, 2 = train, 3 = test, 4 = external, 5 = production
switch = {1: 'demo', 2: 'train', 3: 'test', 4: 'external', 5: 'production'}

while True:
    user_input = input("Enter workflow mode (1-5): ")
    try:
        choice = int(user_input)
        environment = switch[choice]
        break
    except ValueError:
        print("Invalid input. Please enter an integer from 1 to 5.")
    except KeyError:
        print("Invalid choice. Please enter a number from 1 to 5.")

print(f"Selected environment: {environment}")


Selected environment: demo


## Importing Our List of Stocks

The next thing we need to do is import the constituents of the S&P 500.

These constituents change over time, so in an ideal world you would connect directly to the index provider (Standard & Poor's) and pull their real-time constituents on a regular basis.

Paying for access to the index provider's API is outside of the scope of this course. 

There's a static version of the S&P 500 constituents available here. [Click this link to download them now](https://drive.google.com/file/d/1ZJSpbY69DVckVZlO9cC6KkgfSufybcHN/view?usp=sharing). Move this file into the `starter-files` folder so it can be accessed by other files in that directory.

Now it's time to import these stocks to our Jupyter Notebook file.

In [3]:
# getting the right dataset for the right environment
if environment == 'production':
    stocks = pd.read_csv('sp_500_stocks_eodhd.csv')
if environment == 'demo':
    stocks = pd.read_csv('sp_500_stocks_eodhd_demo.csv')

# cleaning the data
stocks['Ticker'] = stocks['Ticker'].astype(str).str.strip()
stocks['Symbol'] = stocks['Symbol'].astype(str).str.strip()
# stocks = stocks[stocks['Symbol'].str.endswith('.US', na=False)]
stocks = stocks.drop_duplicates(subset=['Symbol']).reset_index(drop=True)

## Acquiring an API Token

Now it's time to import our EODHD API token. This is the data provider that we will be using throughout this course.

API tokens (and other sensitive information) should be stored in a `config.py` file that doesn't get pushed to your local Git repository. We'll be using a sandbox API token in this course, which means that the data we'll use is randomly-generated and (more importantly) has no cost associated with it.

[Click here](http://nickmccullum.com/algorithmic-trading-python/secrets.py) to download your `secrets.py` file but rename it to 'config.py'. Move the file into the same directory as this Jupyter Notebook before proceeding.

In [4]:
# getting the right dataset for the right environment
if environment == 'production' or environment == 'external' or environment == 'test' or environment == 'train':
    from config import EODHD_API_KEY
if environment == 'demo':
    EODHD_API_KEY = 'demo'
    

## Making Our First API Calls

Now it’s time to make our first API requests to EODHD.

when `environment == 'test'`:

1. Make a simple API call to get the latest **close price** for `AAPL.US`.
2. Make a second API call to retrieve additional fundamentals for `AAPL.US`:
    - **P/E** *(Price-to-Earnings ratio)*
    - **E/S (EPS)** *(Earnings per Share)*
    - **Market Cap** *(Market Capitalization)*
    - **P/B** *(Price-to-Book ratio)*
    - **P/S** *(Price-to-Sales ratio)*
    - **EV** *(Enterprise Value)*
    - **EBITDA** *(Earnings Before Interest, Taxes, Depreciation, and Amortization)*
    - **Gross Profit** *(Revenue minus Cost of Goods Sold)*

For this project, only the following fields are required to build the equal-weight portfolio:

- **Market capitalization** for each stock  
- **Stock price (close)**

The extra metrics are optional and included for practice.



In [5]:
# only for testing purposes: get price and fundamentals for AAPL.US
if environment == 'demo':
    symbol = 'AAPL.US'

    # get price (close) for AAPL.US from real-time endpoint
    api_url = f'https://eodhistoricaldata.com/api/real-time/{symbol}?api_token={EODHD_API_KEY}&fmt=json'
    response = requests.get(api_url)
    data_price = response.json()
    print('price (close):', data_price['close'])

    # get valuation and highlights sections from fundamentals endpoint
    api_url = f'https://eodhd.com/api/fundamentals/{symbol}'

    # Try to limit payload to the needed sections (if supported by endpoint)
    params = {
        'api_token': EODHD_API_KEY,
        'fmt': 'json',
        'filter': 'Valuation,Highlights,General'
    }
    raw = requests.get(api_url, params=params).json()

    # Read only the required sections/keys
    valuation = raw.get('Valuation', {})
    highlights = raw.get('Highlights', {})
    general = raw.get('General', {})

    snapshot = {
        'Name': general.get('Name'),
        'P/E': valuation.get('TrailingPE'),
        'E/S': highlights.get('EarningsShare'),
        'MarketCap': highlights.get('MarketCapitalization'),
        'P/B': valuation.get('PriceBookMRQ'),
        'P/S': valuation.get('PriceSalesTTM'),
        'EV': valuation.get('EnterpriseValue'),
        'EBITDA': highlights.get('EBITDA'),
        'GrossProfit': highlights.get('GrossProfit')
    }
    print('snapshot:', snapshot)

price (close): 251.9
snapshot: {'Name': 'Apple Inc', 'P/E': 32.7257, 'E/S': 7.91, 'MarketCap': 3804704800768, 'P/B': 42.6034, 'P/S': 8.7341, 'EV': 3780799348800, 'EBITDA': 152901992448, 'GrossProfit': None}


## Parsing Our API Call

The API call that we executed in the last code block contains all of the information required to build our equal-weight S&P 500 strategy. 

With that said, the data isn't in a proper format yet. We need to parse it first.

In [6]:
close_price = data_price['close']
market_cap = snapshot['MarketCap']  # in currency units
company_name = snapshot['Name'] 

## Adding Our Stocks Data to a Pandas DataFrame

The next thing we need to do is add our stock's price and market capitalization to a pandas DataFrame. Think of a DataFrame like the Python version of a spreadsheet. It stores tabular data.

In [7]:
my_columns = ['Primary Ticker', 'Company Name', 'Stock Price', 'Market Capitalization', 'Number of Shares to Buy']
final_dataframe = pd.DataFrame(columns=my_columns)


In [8]:
# only for testing purposes: append AAPL.US data to final dataframe
if environment == 'demo':
    final_dataframe = pd.concat(
    [
        final_dataframe,
        pd.DataFrame(
            [[symbol, company_name, close_price, market_cap, None]],
            columns=my_columns
        )
    ],
    ignore_index=True
)
final_dataframe

,Primary Ticker,Company Name,Stock Price,Market Capitalization,Number of Shares to Buy
0,AAPL.US,Apple Inc,251.9,3804704800768,None


## Looping Through The Tickers in Our List of Stocks

Using the same logic that we outlined above, we can pull data for all S&P 500 stocks and store their data in the DataFrame using a `for` loop.

In [9]:
final_dataframe = pd.DataFrame(columns=my_columns)
for symbol in stocks['Symbol']:  # test dataset has only four stocks, use it for testing

    # Fetch price (close) from real-time data for each stock (similar to AAPL.US example)
    api_url = f'https://eodhistoricaldata.com/api/real-time/{symbol}?api_token={EODHD_API_KEY}&fmt=json'
    response = requests.get(api_url)
    data_price = response.json()

    # Fetch company name and market capitalization from fundamentals data for each stock (similar to AAPL.US example)
    api_url = f'https://eodhd.com/api/fundamentals/{symbol}'

    # Try to limit payload to the needed sections (if supported by endpoint)
    params = {
        'api_token': EODHD_API_KEY,
        'fmt': 'json',
        'filter': 'Highlights,General'
    }
    raw = requests.get(api_url, params=params).json()

    # Read only the required sections/keys
    highlights = raw.get('Highlights', {})
    general = raw.get('General', {})

    snapshot = {
        'Name': general.get('Name'),
        'MarketCap': highlights.get('MarketCapitalization')
    }    

    # Append data to final_dataframe
    final_dataframe = pd.concat(
        [
            final_dataframe,
            pd.DataFrame(
                [[symbol, snapshot['Name'], data_price['close'], snapshot['MarketCap'], None]],
                columns=my_columns
            )
        ],
        ignore_index=True
    )
    print(f'Processed {symbol}')
final_dataframe

Processed AAPL.US
Processed TSLA.US
Processed AMZN.US


,Primary Ticker,Company Name,Stock Price,Market Capitalization,Number of Shares to Buy
0,AAPL.US,Apple Inc,251.900,3804704800768,None
1,TSLA.US,Tesla Inc,343.065,1323932975104,None
2,AMZN.US,Amazon.com Inc,212.770,2284283756544,None


# Review User Data
## Check quota and remaining API calls

In [17]:
if False:  # Only for testing purposes, to check if we can fetch user data
    api_url = f'https://eodhd.com/api/user?api_token={EODHD_API_KEY}&fmt=json'
    user = requests.get(api_url).json()
    if user is None:
        raise ValueError(f'Failed to fetch valid user data.')
    user
else:
    print("Skipping user data fetch in this environment.")


Skipping user data fetch in this environment.


## Using Batch API Calls to Improve Performance

Batch API calls are one of the easiest ways to improve the performance of your code.

This is because HTTP requests are typically one of the slowest components of a script.

Also, API providers will often give you discounted rates for using batch API calls since they are easier for the API provider to respond to.

EODHD limits their batch API calls to 100 tickers per request. Still, this reduces the number of API calls we'll make in this section from 500 to 5 - huge improvement! In this section, we'll split our list of stocks into groups of 100 and then make a batch API call for each group.

In [11]:
# Batch pricing with EODHD real-time endpoint.
# This is much faster than 1 request per symbol for large universes (500+).

def fetch_json_with_meta_local(url, timeout=20):
    try:
        response = requests.get(url, timeout=timeout)
        content_type = (response.headers.get('Content-Type') or '').lower()
        if response.status_code != 200:
            return {
                'ok': False,
                'status_code': response.status_code,
                'reason': 'http_error',
                'content_type': content_type,
                'data': None
            }
        if 'json' not in content_type:
            return {
                'ok': False,
                'status_code': response.status_code,
                'reason': 'non_json_response',
                'content_type': content_type,
                'data': None
            }
        return {
            'ok': True,
            'status_code': response.status_code,
            'reason': 'ok',
            'content_type': content_type,
            'data': response.json()
        }
    except requests.RequestException:
        return {
            'ok': False,
            'status_code': None,
            'reason': 'request_exception',
            'content_type': None,
            'data': None
        }
    except ValueError:
        return {
            'ok': False,
            'status_code': None,
            'reason': 'invalid_json',
            'content_type': None,
            'data': None
        }

def chunked(values, size):
    for i in range(0, len(values), size):
        yield values[i:i + size]

# Use the full available universe from stocks
symbols_to_fetch = stocks['Symbol'].head(len(stocks)).tolist()
# (Equivalent: stocks['Symbol'].tolist())

chunk_size = 100  # conservative size to keep URLs manageable
price_records = []
failed_chunks = 0

for symbol_chunk in chunked(symbols_to_fetch, chunk_size):
    symbols_csv = ','.join(symbol_chunk)
    print(symbols_csv)
    api_url = f'https://eodhd.com/api/real-time/{symbols_csv}?api_token={EODHD_API_KEY}&fmt=json'
    result = fetch_json_with_meta_local(api_url)

    if not result['ok'] or result['data'] is None:
        failed_chunks += 1
        print(f"BATCH FAIL ({result['reason']}, status={result['status_code']}): first symbol {symbol_chunk[0]}")
        continue

    chunk_data = result['data']
    if isinstance(chunk_data, dict):
        chunk_data = [chunk_data]

    for item in chunk_data:
        code = item.get('code')
        close = item.get('close')
        if code is None or close is None:
            continue
        price_records.append({'Symbol': code, 'Stock Price': close})

batch_prices_df = pd.DataFrame(price_records).drop_duplicates(subset=['Symbol'])
batch_universe_df = stocks[stocks['Symbol'].isin(symbols_to_fetch)][['Ticker', 'Symbol']].copy()
batch_universe_df = batch_universe_df.merge(batch_prices_df, on='Symbol', how='left')

print(f'Requested symbols: {len(symbols_to_fetch)}')
print(f'Chunks sent: {math.ceil(len(symbols_to_fetch)/chunk_size)} | Failed chunks: {failed_chunks}')
print(f'Prices received: {batch_universe_df["Stock Price"].notna().sum()}')
batch_universe_df.head()

AAPL.US,TSLA.US,AMZN.US
Requested symbols: 3
Chunks sent: 1 | Failed chunks: 0
Prices received: 3


,Ticker,Symbol,Stock Price
0,AAPL,AAPL.US,251.900
1,TSLA,TSLA.US,343.065
2,AMZN,AMZN.US,212.770


In [12]:
# Fundamentals fetch (one request per symbol), as bulk requests are available only via Extended Fundamentals Plan
# Pull Name from General and MarketCap from Highlights for each symbol.

symbols_to_fetch = stocks['Symbol'].head(len(stocks)).tolist()

fundamental_records = []
failed_symbols = 0

for symbol in symbols_to_fetch:
    api_url = f'https://eodhd.com/api/fundamentals/{symbol}?api_token={EODHD_API_KEY}&fmt=json'
    result = fetch_json_with_meta_local(api_url)

    if not result['ok'] or result['data'] is None:
        failed_symbols += 1
        print(f"FUNDAMENTALS FAIL ({result['reason']}, status={result['status_code']}): {symbol}")
        continue

    raw = result['data']
    general = raw.get('General') or {}
    highlights = raw.get('Highlights') or {}

    fundamental_records.append({
        'Symbol': symbol,
        'Name': general.get('Name'),
        'MarketCap': highlights.get('MarketCapitalization')
    })

batch_fundamentals_df = pd.DataFrame(
    fundamental_records,
    columns=['Symbol', 'Name', 'MarketCap']
).drop_duplicates(subset=['Symbol'])
batch_fundamentals_universe_df = stocks[stocks['Symbol'].isin(symbols_to_fetch)][['Ticker', 'Symbol']].copy()
batch_fundamentals_universe_df = batch_fundamentals_universe_df.merge(batch_fundamentals_df, on='Symbol', how='left')

print(f'Requested symbols: {len(symbols_to_fetch)}')
print(f'Failed symbols: {failed_symbols}')
print(f'Names received: {batch_fundamentals_universe_df["Name"].notna().sum()}')
print(f'Market caps received: {batch_fundamentals_universe_df["MarketCap"].notna().sum()}')
batch_fundamentals_universe_df.head()

Requested symbols: 3
Failed symbols: 0
Names received: 3
Market caps received: 3


,Ticker,Symbol,Name,MarketCap
0,AAPL,AAPL.US,Apple Inc,3804704800768
1,TSLA,TSLA.US,Tesla Inc,1323932975104
2,AMZN,AMZN.US,Amazon.com Inc,2284283756544


In [13]:
# Combine batch price data + batch fundamentals into final dataframe schema.
combined_batch_df = batch_universe_df.merge(
    batch_fundamentals_universe_df[['Symbol', 'Name', 'MarketCap']],
    on='Symbol',
    how='left'
 )

final_dataframe = pd.DataFrame({
    'Primary Ticker': combined_batch_df['Symbol'],
    'Company Name': combined_batch_df['Name'],
    'Stock Price': combined_batch_df['Stock Price'],
    'Market Capitalization': combined_batch_df['MarketCap'],
    'Number of Shares to Buy': None
})[my_columns]

print(f'Combined rows: {len(final_dataframe)}')
print(f'Stock prices available: {final_dataframe["Stock Price"].notna().sum()}')
print(f'Market caps available: {final_dataframe["Market Capitalization"].notna().sum()}')
final_dataframe.head()

Combined rows: 3
Stock prices available: 3
Market caps available: 3


,Primary Ticker,Company Name,Stock Price,Market Capitalization,Number of Shares to Buy
0,AAPL.US,Apple Inc,251.900,3804704800768,None
1,TSLA.US,Tesla Inc,343.065,1323932975104,None
2,AMZN.US,Amazon.com Inc,212.770,2284283756544,None


## Calculating the Number of Shares to Buy

As you can see in the DataFrame above, we stil haven't calculated the number of shares of each stock to buy.

We'll do that next.

In [14]:
portfolio_size = input("Enter the value of your portfolio: ")

try:
    val = float(portfolio_size)
except ValueError:
    print("Invalid input. Please enter a numeric value for the portfolio size.")
    portfolio_size = input("Enter the value of your portfolio: ")
    val = float(portfolio_size)

print(f'Portfolio size set to: {val:,.2f}')

Portfolio size set to: 50,000.00


In [15]:
portfolio_size = val/len(final_dataframe.index) 
print(f'Equal weight per stock: {portfolio_size:,.2f}')
for i in range(0, len(final_dataframe.index)):
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(portfolio_size / final_dataframe.loc[i, 'Stock Price'])
final_dataframe.head()

Equal weight per stock: 16,666.67


,Primary Ticker,Company Name,Stock Price,Market Capitalization,Number of Shares to Buy
0,AAPL.US,Apple Inc,251.900,3804704800768,66
1,TSLA.US,Tesla Inc,343.065,1323932975104,48
2,AMZN.US,Amazon.com Inc,212.770,2284283756544,78


## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

### Initializing our XlsxWriter Object

In [20]:
writer = pd.ExcelWriter('recommended_trades.xlsx', engine='xlsxwriter')
final_dataframe.to_excel(writer, sheet_name='Recommended Trades', index=False)

### Creating the Formats We'll Need For Our `.xlsx` File

Formats include colors, fonts, and also symbols like `%` and `$`. We'll need four main formats for our Excel document:
* String format for tickers
* \\$XX.XX format for stock prices
* \\$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

In [21]:
background_color = '#0a0a23'
font_color = '#ffffff'

string_format = writer.book.add_format(
    {
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)
number_format = writer.book.add_format(
    {
        'num_format': '#,##0.00',
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)
integer_format = writer.book.add_format(
    {
        'num_format': '#,##0',
        'font_color': font_color,
        'bg_color': background_color,
        'border': 1
    }
)

### Applying the Formats to the Headers and Columns of Our `.xlsx` File

In the next two code cells, we format the Excel worksheet in a structured way:

- First, we create a `column_formats` dictionary that maps each Excel column (`A` to `E`) to:
    1. the header label,  
    2. the cell format object, and  
    3. the desired column width.

- Next, we loop through this dictionary twice:
    - once to write styled header cells in row 1 using `write(...)`,  
    - and once to apply width and default formatting to each full column using `set_column(...)`.

This keeps formatting centralized, consistent, and easy to maintain.


In [33]:
# Set column widths and formats
column_formats = {
    'A': ['Primary Ticker', string_format, 20],
    'B': ['Company Name', string_format, 40],
    'C': ['Stock Price', number_format, 20],
    'D': ['Market Capitalization', number_format, 25],
    'E': ['Number of Shares to Buy', integer_format, 25]
}

In [34]:
# Apply formats to header
for column, (header, _, _) in column_formats.items():
    writer.sheets['Recommended Trades'].write(f'{column}1', header, string_format)

# Apply formats to columns
for column in column_formats:
    writer.sheets['Recommended Trades'].set_column(
        f'{column}:{column}',
        column_formats[column][2],
        column_formats[column][1]
    )


## Saving Our Excel Output

The next code cell defines and calls a helper function, `finalize_excel_writer(writer)`, to safely finish the Excel export.

It does three things:

- Validates that `writer` exists before attempting to save.
- Detects whether the workbook is already closed (common in notebook re-runs) and exits cleanly.
- Uses `writer.close()` as the preferred modern API, with `writer.save()` only as a backward-compatibility fallback.

This is superior to calling `writer.save()` directly because it is more **robust** and **future-proof**:

- `close()` is the recommended pandas interface.
- It prevents duplicate-close/save issues in iterative notebook execution.
- It supports both newer and older pandas behaviors without changing your workflow.

In [ ]:
# Future-proof save step for pandas ExcelWriter
def finalize_excel_writer(excel_writer):
    if excel_writer is None:
        raise ValueError("`writer` is not available.")

    # Safe for repeated execution in notebooks
    if bool(getattr(getattr(excel_writer, 'book', None), 'fileclosed', False)):
        print("Workbook is already closed.")
        return

    # Preferred API in modern pandas; fallback kept for compatibility
    close_method = getattr(excel_writer, 'close', None)
    if callable(close_method):
        close_method()
    else:
        excel_writer.save()

    print("Excel file saved successfully.")

finalize_excel_writer(writer)